# Lab 2.4 (No Key) - Function Calling and a ReAct Agent (Start)
**Day 2 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_4_agent_nokey/start/lab2_4_agent_nokey.ipynb)

**Goal:** Build the function-calling and ReAct loop by hand: define tools, let a planner request them, run each tool, and feed the result back until the goal is met.

> **How this runs without an API key.** This version replaces the LLM with a small scripted planner so the function-calling and ReAct loop are fully visible and deterministic. It is pure Python, no installs, no key, no downloads. In the API lab the model performs the planning step; the surrounding loop is identical.

Look for the `# TODO` markers and fill them in. The `solution/` notebook has the full answer if you get stuck.

## Setup
Nothing to install and no API key. This lab is pure Python and runs instantly.

## 1. Define two tools
A tool is a plain function. `get_tax_rate` is given. Write the second one.

In [ ]:
def get_tax_rate(state_code: str) -> float:
    """Return the sales tax rate as a percent for a US state."""
    rates = {"CA": 7.25, "NY": 4.0, "TX": 6.25, "WA": 6.5, "FL": 6.0}
    return rates.get(state_code.upper(), 0.0)


# TODO: Write calculate_total(amount: float, tax_rate_percent: float) -> float
# that returns the amount plus tax, rounded to 2 decimals.
# Hint: return round(amount * (1 + tax_rate_percent / 100.0), 2)


print("Sanity check:", calculate_total(250.0, get_tax_rate("CA")))

## 2. Register tools and a planner
`TOOLS` maps names to functions, the way a model looks up a tool by name. `plan_next_step` is the scripted stand-in for the LLM. You do not need to change these cells.

In [ ]:
TOOLS = {"get_tax_rate": get_tax_rate, "calculate_total": calculate_total}

In [ ]:
def plan_next_step(question, observations):
    """Stand-in for the LLM. Returns the next action, or the final answer.

    A real agent lets the model produce this from `question` and the
    observations so far. Here we script it so the loop is deterministic.
    """
    if len(observations) == 0:
        return {"type": "action",
                "thought": "I need California's sales tax rate first.",
                "tool": "get_tax_rate",
                "args": {"state_code": "CA"}}
    if len(observations) == 1:
        return {"type": "action",
                "thought": "Now add that tax to the $250 purchase.",
                "tool": "calculate_total",
                "args": {"amount": 250.0, "tax_rate_percent": observations[0]}}
    return {"type": "final",
            "answer": f"The total is ${observations[1]:.2f}."}

## 3. Run the ReAct loop
The loop asks the planner for the next step, runs the requested tool, and records the observation. Fill in the tool-execution step.

In [ ]:
def run_agent(question, max_steps=5):
    print(f"Goal: {question}\n")
    observations = []
    for step in range(1, max_steps + 1):
        decision = plan_next_step(question, observations)
        if decision["type"] == "final":
            return decision["answer"]
        print(f"Step {step}")
        print(f"  Thought    : {decision['thought']}")
        print(f"  Action     : {decision['tool']}({decision['args']})")
        # TODO: run the requested tool and capture its result.
        # Hint: look the function up in TOOLS by name and call it with **args.
        observation = ...
        print(f"  Observation: {observation}")
        observations.append(observation)
    return "Stopped: hit the step limit."

In [ ]:
answer = run_agent(
    "What is the total cost of a $250 purchase in California after sales tax?"
)
print("\nFinal answer:", answer)

The total should be about $268.12 (250 plus 7.25% tax). The trace above is the ReAct loop: think, act with a tool, observe, repeat.

## Stretch goals
Hints only here; the full code is in the solution notebook.

1. **Add a tool** (for example a flat shipping fee) and extend the planner to use it.
2. **Guardrails.** Validate tool arguments before running, and stop if a tool name is unknown.

In [ ]:
# TODO (stretch): your experiments here.